### Tools

###### Models cam request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairing of:

###### 1. A schema including the name of the tool, a description, and/or arguments definitions ( often a JSON schema)
###### 2. A function or coroutine to execute.

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROOG_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")
response = model.invoke("Why do parrots talk?")
response

In [3]:
### creating tools

from langchain.tools import tool

### We will use @Tool decorator and that function become tool

@tool
def get_weather(location:str)-> str:
    """
     Get the weather at a location

    """
    return  f"It's Sunny in {location}"


### Bind the tools
model_with_tools = model.bind_tools([get_weather])

In [ ]:
### Invoke the tools

response = model_with_tools.invoke("What is the weather in Kansas?")
print(response)
for tool_call in response.tool_calls:
    # view tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

### Tool Execution Loops

In [ ]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content":"What's the weather in Kansas?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute the tool with generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The Current weather in Kansas is 72 degree F and Sunny"